# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jahnzaibakhtar/Flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
One row = one client-content-day of performance: report_date × client_id × content_id, from
fact_content_daily_performance, restricted to month=2026-03 — a mid-panel month, not the sealed
final-month _sample (June 2026 is reserved as the outcome window for any past→future label, per
the panel warning). I verify this grain claim, its row count/date span, its missingness pattern,
and its availability filter in Section 3 below.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
Feature:
- ctr — trailing measured rate, known same-day.
- gsc_avg_position — Search Console reports it same-day.
- scroll_rate — measured from completed sessions, known same-day.
- ai_traffic_pct — same-day traffic-source split.
- content_type — static attribute set at publish time.

Label / proxy:
- No trained label yet. is_declining_label is a candidate but it's rule-derived (from
  trend_direction, itself from trend_pct) — not an observed outcome, so it's excluded below,
  not used here. The real target (an observed future-window outcome) isn't built yet.

Context (not a feature, not a label, but needed to interpret rows):
- client_id, content_id — for grouping, joining, and per-client filtering only; never features
  (pseudonyms).
- report_date — needed to define the window, not a predictive signal itself.
- ga4_data_available, ga4_data_start, gsc_data_start — needed to filter rows correctly.

Excluded:
- trend_direction and trend_pct — is_declining_label is computed from them; using either as a
  feature would let the model reconstruct the label's own formula, not learn a real pattern.
- fact_content_daily_performance_sample — the final month, the natural outcome window for any
  label I'll eventually build; using it now means developing inside my own future test window.
- GA4 columns on rows where ga4_data_available is FALSE — zero-filled placeholders, not real
  "zero engagement."
- fact_content_query_90d — excluded from this month's slice entirely for now. Its per-content
  context columns repeat on every row (must be read with ANY_VALUE, never SUM, or context gets
  double-counted), and its 90-day window overlaps the panel's final months — joining it in
  before checking window alignment would risk pulling future-window information into a
  present-moment feature.

Trap check on dim_clients: it only accrues from each client's registration/data-start day —
rows before that aren't zero-engagement, they're absent history. This is why the availability
filter in Section 3 is a correctness requirement, not just cleanup.

Output: a per-content-day row of feature values (ctr, gsc_avg_position, scroll_rate,
ai_traffic_pct, content_type) for March 2026, filtered to rows with real GA4 availability —
handed off as the feature frame for opportunity scoring, not yet a trained prediction.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
Verify grain: one row really is one report_date  client_id  content_id.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download
import os

token = os.environ["HF_TOKEN"]  # from Colab Secret, never pasted literal

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data.parquet",
    token=token,
)
df_month = pd.read_parquet(path)

grain_cols = ["report_date", "client_id", "content_id"]
dupe_counts = df_month.groupby(grain_cols).size()
print("Groups with >1 row (should be 0 if grain claim holds):", (dupe_counts > 1).sum())

Verify missingness — is it random, or does it follow a pattern (e.g. by client)?

In [ ]:
missing_by_client = df_month.groupby("client_id")["gsc_avg_position"].apply(
    lambda s: (s == 0).mean()  # 0 is a "no data" sentinel, not a real rank
)
print("Share of avg_position==0 rows, top 5 clients by missingness:")
print(missing_by_client.sort_values(ascending=False).head())

Verify availability — filter GA4 rows with IS TRUE, show survivors.

clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients/data.parquet",
    token=token,
)
dim_clients = pd.read_parquet(clients_path)

merged = df_month.merge(dim_clients, on="client_id", how="left")
before = len(merged)
available = merged[merged["ga4_data_available"] == True]  # IS TRUE filter
after = len(available)
print(f"Rows before availability filter: {before}")
print(f"Rows after ga4_data_available IS TRUE: {after}")
print(f"Dropped: {before - after} ({(1 - after/before):.1%})")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
This data can never tell me why a page's traffic changed — only that it did, across whichever
signals I measured. It can't support causal claims like "refreshing this page caused recovery."

History depth is unbalanced across clients: gsc_data_start varies widely, so a client with only
2 months of history looks artificially "stable" or "sparse" next to one with the full 17-month
panel — comparing raw trends across clients without checking dim_clients.gsc_data_start first
would be misleading.

Early rows for some clients are GSC-only: GA4 columns are zero-filled with
ga4_data_available=FALSE before a client's ga4_data_start, so my March 2026 slice may still
contain clients with no real engagement data for that window — confirmed by the missingness
query above, not assumed from the full panel.

Window overlap: fact_content_query_90d's 90-day window overlaps the final months of the whole
panel, so if my eventual label lives in the last 30 days of a client's history, only
*_prev30-style columns from that table would be safe features — anything else risks pulling
information from inside the outcome window itself. I've excluded this table from the current
slice for that reason (see Section 2).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.